In [ ]:
import os
os.environ['HF_HOME']='/workspace/AAIPL/hf_models/'
from unsloth import FastLanguageModel
import torch

# ═══ Configuration ═══
max_seq_length = 2048   # A-Agent: max_new_tokens=512, but input can be long
dtype = torch.bfloat16  # BF16 for ROCm / MI300X
load_in_4bit = False    # Full precision — we have 192GB VRAM

# Qwen2.5-14B-Instruct via Unsloth
model_name = "unsloth/Qwen2.5-14B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    trust_remote_code=True,
)

print(f"Model loaded: {model_name}")
print(f"Parameter count: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                        # LoRA rank (higher = more capacity)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,               # 2x rank for faster convergence
    lora_dropout=0,              # 0 is optimized by Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
import json
from datasets import Dataset

# ═══ Load dataset ═══
DATA_DIR = "/workspace/AAIPL/hf_models/hub/datasets--Aayushktyagi--SFT_Apti/snapshots/f5199fdcda9db63487aeca763201aa4c05ef11f2"  # Adjust path as needed on remote server

with open(f"{DATA_DIR}/aagent_chatml_train.json") as f:
    train_data = json.load(f)
with open(f"{DATA_DIR}/aagent_chatml_val.json") as f:
    val_data = json.load(f)

print(f"Train: {len(train_data):,} samples")
print(f"Val:   {len(val_data):,} samples")
print(f"Sample keys: {list(train_data[0].keys())}")
print(f"\nSample messages[0]:")
for msg in train_data[0]['messages']:
    print(f"  [{msg['role']}]: {msg['content'][:100]}...")

In [ ]:
# Convert to HuggingFace Datasets
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train dataset: {train_dataset}")
print(f"Val dataset:   {val_dataset}")

In [ ]:
batch = val_dataset[0]
batch.keys()

In [ ]:
batch['messages']

In [ ]:
from unsloth.chat_templates import standardize_sharegpt

# Standardize message format (handles role name normalization)
train_dataset = standardize_sharegpt(train_dataset)
val_dataset = standardize_sharegpt(val_dataset)

# Formatting function: apply model's native chat template
def formatting_prompts_func(examples):
    # standardize_sharegpt may rename 'messages' to 'conversations'
    key = "conversations" if "conversations" in examples else "messages"
    convos = examples[key]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

# Preview formatted text
print("═" * 60)
print("Sample formatted training text:")
print("═" * 60)
print(train_dataset["text"][0][:500])
print("...")

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only
from transformers import DataCollatorForSeq2Seq

# ═══ Training Configuration ═══
OUTPUT_DIR = "aagent_qwen25_outputs"
NUM_EPOCHS = 1        # Start with 1, increase if val loss still dropping
BATCH_SIZE = 128       # MI300X can handle more; tune based on seq_length
GRAD_ACCUM = 8        # Effective batch = 128 * 8 = 1024
LEARNING_RATE = 2e-4

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    packing=False,  # Don't pack — preserve conversation boundaries
    args=SFTConfig(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=10,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
        bf16=True,
        dataloader_pin_memory=False,
        gradient_checkpointing=True,
        dataloader_num_workers=0,  # For ROCm stability
        remove_unused_columns=True,
    ),
)

# Train ONLY on assistant responses — mask system + user tokens from loss
# NOTE: Qwen2.5 uses ChatML format: <|im_start|>role\ncontent<|im_end|>
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print(f"Training config:")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Total train samples: {len(train_dataset):,}")
print(f"  Total val samples:   {len(val_dataset):,}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Steps/epoch: ~{len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}")

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total_mem = round(gpu_stats.total_memory / 1024**3, 2)
print(f"GPU: {gpu_stats.name}")
print(f"Total VRAM: {total_mem} GB")
print(f"Reserved before training: {reserved} GB")
print(f"Available for KV cache + training: {total_mem - reserved:.1f} GB")

In [ ]:
# Switch to training mode
FastLanguageModel.for_training(model)

# Train
trainer_stats = trainer.train()

# Print final stats
print(f"\nTraining completed!")
print(f"  Runtime: {trainer_stats.metrics['train_runtime']:.0f}s ({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"  Final loss: {trainer_stats.metrics.get('train_loss', 'N/A')}")
peak_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"  Peak VRAM: {peak_mem} GB / {total_mem} GB ({100*peak_mem/total_mem:.1f}%)")

In [ ]:
# Save LoRA adapters (small, ~100-300MB)
LORA_PATH = "aagent_qwen25_lora"
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f"LoRA adapters saved to: {LORA_PATH}")

# Save merged 16-bit model (full size, for vLLM deployment)
MERGED_PATH = "aagent_qwen25_merged_16bit"
print(f"Saving merged 16-bit model to: {MERGED_PATH} ...")
model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to: {MERGED_PATH}")

In [ ]:
# ═══ Load trained model (for fresh session / separate eval) ═══
# Skip this cell if model is already in memory from training above.
# Choose ONE of the two methods below:

# ── Method 1: Load base + LoRA adapters (smaller files, fast load) ──
from unsloth import FastLanguageModel
import torch

LORA_PATH = "aagent_qwen25_lora"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_PATH,
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    trust_remote_code=True,
)
print(f"✓ Loaded LoRA model from: {LORA_PATH}")

# # ── Method 2: Load merged 16-bit model (full weights, no adapter overhead) ──
# MERGED_PATH = "aagent_qwen25_merged_16bit"
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=MERGED_PATH,
#     max_seq_length=2048,
#     dtype=torch.bfloat16,
#     load_in_4bit=False,
#     trust_remote_code=True,
# )
# print(f"✓ Loaded merged model from: {MERGED_PATH}")

# Switch to fast inference mode
FastLanguageModel.for_inference(model)
print(f"Parameter count: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model device: {next(model.parameters()).device}")

In [ ]:
# Switch to inference mode (2x faster)
FastLanguageModel.for_inference(model)

# Test questions — one per category
test_questions = [
    # Syllogism
    """Statements:
1. All dogs are animals.
2. All animals are living beings.
Conclusions:
I. All dogs are living beings.
II. Some living beings are dogs.

A) Only conclusion I follows
B) Only conclusion II follows
C) Both conclusions I and II follow
D) Neither conclusion I nor II follows""",

    # Blood Relations
    """A is the father of B. B is the mother of C. D is the brother of A. What is D to C?

A) Grandfather
B) Uncle
C) Great Uncle
D) Father""",

    # Seating Arrangement
    """5 persons A, B, C, D, E are seated in a row (left to right). 
B is to the immediate right of A. 
D is at one of the ends. 
C is to the immediate left of E.
Who is in the middle?

A) A
B) B
C) C
D) E""",

    # Series
    """Find the next term in the series: 2, 6, 18, 54, ?

A) 108
B) 162
C) 180
D) 216""",
]

# System prompt matching the training data
system_prompt = """You are a logical reasoning expert. Answer the given multiple-choice question.
Output constraint: 
You must answer the question and output ONLY valid JSON.

JSON schema:
{
  "properties": {
    "reasoning": {
      "title": "Reasoning",
      "type": "string"
    },
    "answer": {
      "enum": [
        "A",
        "B",
        "C",
        "D"
      ],
      "title": "Answer",
      "type": "string"
    }
  },
  "required": [
    "reasoning",
    "answer"
  ],
  "title": "Answer",
  "type": "object"
}"""

for i, q in enumerate(test_questions):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )
    print(f"\n{'═'*50}")
    print(f"Q{i+1}: {q[:80]}...")
    print(f"A{i+1}: {response[:300]}")

In [ ]:
import json
from collections import defaultdict
import numpy as np

# ═══ Full Validation Accuracy (BATCHED) + Confusion Metrics ═══
with open(f"{DATA_DIR}/aagent_chatml_val.json") as f:
    val_raw = json.load(f)

print(f"Evaluating on FULL validation set: {len(val_raw)} samples\n")

FastLanguageModel.for_inference(model)

EVAL_BATCH_SIZE = 32  # Tune based on VRAM headroom
CLASSES = ["A", "B", "C", "D"]
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

# ── Structured output: JSON schema enforcement (like API response_format) ──
from pydantic import BaseModel, Field
from typing import Literal

class AAgentResponse(BaseModel):
    """Expected A-Agent output schema."""
    reasoning: str
    answer: Literal["A", "B", "C", "D"]

json_processor = None
try:
    from outlines.processors import JSONLogitsProcessor
    json_processor = JSONLogitsProcessor(
        schema=AAgentResponse,
        tokenizer=tokenizer,
        whitespace_pattern=r"[\n\t ]*",
    )
    print("✓ Using outlines JSONLogitsProcessor (structured JSON output enforced)")
except ImportError:
    print("⚠ outlines not installed — pip install outlines")
    print("  Falling back to unconstrained generation + post-hoc JSON parsing")
except Exception as e:
    print(f"⚠ outlines init failed: {e}")
    print("  Falling back to unconstrained generation + post-hoc JSON parsing")

# ── Generation config ──
GEN_CONFIG = dict(
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    top_p=0.9,
    repetition_penalty=1.2,
)

# ── Step 1: Pre-process all samples ──
samples = []
skipped = 0

for item in val_raw:
    msgs = item["messages"]
    gt_response = msgs[-1]["content"]
    try:
        gt_parsed = json.loads(gt_response)
        gt_answer = gt_parsed["answer"]
    except Exception:
        skipped += 1
        continue

    user_content = msgs[1]["content"] if len(msgs) > 1 else ""
    if "syllogism" in user_content.lower() or "Statements:" in user_content:
        topic_key = "Syllogisms"
    elif "blood" in user_content.lower() or "father" in user_content.lower() or "mother" in user_content.lower():
        topic_key = "Blood Relations"
    elif "seated" in user_content.lower() or "seating" in user_content.lower():
        topic_key = "Seating Arrangements"
    elif "series" in user_content.lower() or "next term" in user_content.lower() or "Find the" in user_content:
        topic_key = "Series"
    else:
        topic_key = "Other"

    samples.append({
        "gt_answer": gt_answer,
        "topic_key": topic_key,
        "prompt_msgs": msgs[:-1],
    })

print(f"Valid samples: {len(samples)} (skipped {skipped} with unparseable GT)")
print(f"Generation config: {GEN_CONFIG}")
print(f"JSON enforcement: {'outlines JSONLogitsProcessor' if json_processor else 'post-hoc parsing'}")

# ── Step 2: Left-pad tokenizer for batched generation ──
orig_padding_side = tokenizer.padding_side
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ── Step 3: Batched inference ──
correct = 0
total = 0
parse_fails = 0
topic_stats = defaultdict(lambda: {"correct": 0, "total": 0})

confusion = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
topic_confusion = defaultdict(lambda: np.zeros((len(CLASSES), len(CLASSES)), dtype=int))

num_batches = (len(samples) + EVAL_BATCH_SIZE - 1) // EVAL_BATCH_SIZE

for batch_idx in range(num_batches):
    start = batch_idx * EVAL_BATCH_SIZE
    end = min(start + EVAL_BATCH_SIZE, len(samples))
    batch = samples[start:end]

    prompt_texts = [
        tokenizer.apply_chat_template(
            s["prompt_msgs"], add_generation_prompt=True, tokenize=False
        )
        for s in batch
    ]
    inputs = tokenizer(
        prompt_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_seq_length,
    ).to(model.device)

    prompt_lengths = inputs["input_ids"].shape[1]

    gen_kwargs = dict(
        **inputs,
        **GEN_CONFIG,
        pad_token_id=tokenizer.pad_token_id,
    )
    if json_processor is not None:
        gen_kwargs["logits_processor"] = [json_processor]

    outputs = model.generate(**gen_kwargs)

    for i, s in enumerate(batch):
        response = tokenizer.decode(
            outputs[i][prompt_lengths:], skip_special_tokens=True
        )

        pred = None
        try:
            parsed = json.loads(response)
            pred = parsed["answer"]
        except Exception:
            for letter in CLASSES:
                if f'"answer": "{letter}"' in response or f'"answer":"{letter}"' in response:
                    pred = letter
                    break
            if pred is None:
                for ch in response.strip()[:20]:
                    if ch in "ABCD":
                        pred = ch
                        break

        if pred and pred in class_to_idx and s["gt_answer"] in class_to_idx:
            total += 1
            gt_idx = class_to_idx[s["gt_answer"]]
            pred_idx = class_to_idx[pred]
            confusion[gt_idx][pred_idx] += 1
            topic_confusion[s["topic_key"]][gt_idx][pred_idx] += 1
            topic_stats[s["topic_key"]]["total"] += 1
            if pred == s["gt_answer"]:
                correct += 1
                topic_stats[s["topic_key"]]["correct"] += 1
        else:
            parse_fails += 1

    processed = end
    if processed % 500 < EVAL_BATCH_SIZE or batch_idx == num_batches - 1:
        acc = 100 * correct / total if total > 0 else 0
        print(f"  [{processed}/{len(samples)}] accuracy: {correct}/{total} = {acc:.1f}%  parse_fails: {parse_fails}")

tokenizer.padding_side = orig_padding_side

# ═══════════════════════════════════════════════════════════════
# RESULTS
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═'*60}")
print(f"FULL VALIDATION RESULTS ({len(val_raw)} samples)")
print(f"{'═'*60}")
print(f"Overall Accuracy: {correct}/{total} = {100*correct/total:.2f}%")
print(f"Parse failures: {parse_fails}  |  GT skipped: {skipped}")

# ── Per-Topic Breakdown ──
print(f"\n{'─'*60}")
print(f"Per-Topic Accuracy:")
print(f"{'─'*60}")
for topic, stats in sorted(topic_stats.items()):
    t_acc = 100 * stats["correct"] / stats["total"] if stats["total"] > 0 else 0
    print(f"  {topic:25s}: {stats['correct']:5d}/{stats['total']:5d} = {t_acc:.1f}%")

# ── Overall Confusion Matrix ──
print(f"\n{'─'*60}")
print(f"Confusion Matrix (rows=GT, cols=Pred):")
print(f"{'─'*60}")
header = "       " + "".join(f"  Pred {c}" for c in CLASSES)
print(header)
for r, cls in enumerate(CLASSES):
    row_str = f"  GT {cls}: " + "".join(f"{confusion[r][c]:7d}" for c in range(len(CLASSES)))
    print(row_str)

# ── Per-Class Precision / Recall / F1 ──
print(f"\n{'─'*60}")
print(f"Per-Class Metrics:")
print(f"{'─'*60}")
print(f"  {'Class':>5s}  {'Precision':>9s}  {'Recall':>9s}  {'F1':>9s}  {'Support':>7s}")
macro_p, macro_r, macro_f1 = 0, 0, 0
n_classes_present = 0
for idx, cls in enumerate(CLASSES):
    tp = confusion[idx][idx]
    fn = confusion[idx].sum() - tp
    fp = confusion[:, idx].sum() - tp
    support = confusion[idx].sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"  {cls:>5s}  {precision:>9.4f}  {recall:>9.4f}  {f1:>9.4f}  {support:>7d}")
    if support > 0:
        macro_p += precision
        macro_r += recall
        macro_f1 += f1
        n_classes_present += 1

if n_classes_present > 0:
    print(f"  {'Macro':>5s}  {macro_p/n_classes_present:>9.4f}  {macro_r/n_classes_present:>9.4f}  {macro_f1/n_classes_present:>9.4f}")

# ── Per-Topic Confusion Matrices + Class Metrics ──
print(f"\n{'═'*60}")
print(f"PER-TOPIC CONFUSION MATRICES & CLASS METRICS")
print(f"{'═'*60}")

for topic in sorted(topic_confusion.keys()):
    cm = topic_confusion[topic]
    t_total = cm.sum()
    t_correct = sum(cm[i][i] for i in range(len(CLASSES)))
    t_acc = 100 * t_correct / t_total if t_total > 0 else 0

    print(f"\n┌─ {topic} (n={t_total}, acc={t_acc:.1f}%) ─┐")
    print("       " + "".join(f"  Pred {c}" for c in CLASSES))
    for r, cls in enumerate(CLASSES):
        row_str = f"  GT {cls}: " + "".join(f"{cm[r][c]:7d}" for c in range(len(CLASSES)))
        print(row_str)

    print(f"  {'Class':>5s}  {'Prec':>6s}  {'Rec':>6s}  {'F1':>6s}  {'N':>5s}")
    for idx, cls in enumerate(CLASSES):
        tp = cm[idx][idx]
        fn = cm[idx].sum() - tp
        fp = cm[:, idx].sum() - tp
        support = cm[idx].sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        if support > 0:
            print(f"  {cls:>5s}  {prec:>6.3f}  {rec:>6.3f}  {f1:>6.3f}  {support:>5d}")